In [6]:
## 260929

import sys
import os

sys.path.append('.')

import numpy as np
import torch
import torch.nn.functional as F
import pickle

from torch.amp import autocast
from tqdm import tqdm

import matplotlib.pyplot as plt

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer, train_bpe
from storybot.utils import get_device, generate

In [7]:
def get_lr(it, max_lr, warmup_iters, max_iters):
    if it < warmup_iters:
        return max_lr * (it / warmup_iters)

    if it < max_iters:
        progress = (it - warmup_iters) / (max_iters - warmup_iters)
        return max_lr * (1.0 - progress)

    return 0.0

def get_batch(data, context_len, batch_size, device, random = True, offset = 0):
    if random:
        ix = torch.randint(len(data) - context_len - 1, (batch_size, ))

    else:
        ix = torch.arange(offset, offset + batch_size * context_len, context_len)
        ix = ix[ix + context_len + 1 < len(data)]

        if len(ix) == 0:
            return None, None

    x = torch.stack([torch.from_numpy(data[i:i + context_len].astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy(data[i + 1:i + context_len + 1].astype(np.int64)) for i in ix])

    return x.to(device), y.to(device)

def evaluate(model, val_data, context_len, batch_size, device):
    model.eval()
    total_loss = 0.0
    total_tokens = 0

    max_start = len(val_data) - context_len - 1
    num_batches = (max_start // context_len) // batch_size + 1

    with torch.no_grad():
        for batch_idx in tqdm(range(num_batches), desc = 'Validation'):
            offset = batch_idx * batch_size * context_len
            x, y = get_batch(val_data, context_len, batch_size, device, random = False, offset = offset)

            if x is None:
                break

            with autocast(device_type = device.type, dtype = torch.bfloat16):
                logits = model(x)
                loss = F.cross_entropy(logits.view(-1, logits.size(-1)), y.view(-1), reduction = 'sum')

            total_loss += loss.item()
            total_tokens += y.numel()

    model.train()

    return total_loss, total_tokens

In [8]:
device = get_device()
print(device)

cuda


In [ ]:
vocab_size = 10000
file_path = './tiny_stories_train.txt'

merge_rules = train_bpe(file_path, vocab_size, num_processes = 8)

with open('./merge_rules.pkl', 'wb') as f:
    pickle.dump(merge_rules, f)

Pretokenizing:   0%|          | 0/8 [00:00<?, ?it/s]

In [ ]:
merge_rules_path = './merge_rules.pkl'

tokenizer = BPETokenizer.load_from(merge_rules_path)

tokenizer.encode_file(
    './tiny_stories_train.txt',
    './tiny_stories_train.bin', num_processes = 8, num_chunks = 64
)

tokenizer.encode_file(
    './tiny_stories_valid.txt',
    './tiny_stories_valid.bin', num_processes = 8, num_chunks = 16
)

Encoding chunks: 100%|██████████| 16/16 [00:00<00:00, 18.12it/s]


5465873

In [ ]:
device = get_device()
print(device)

data_path = './tiny_stories_train.bin'
val_data_path = './tiny_stories_valid.bin'
tokenizer_path = './merge_rules.pkl'
model_save_path = './model_pretrain.pt'

context_len = 256
vocab_size = 10000
batch_size = 32
learning_rate = 0.001
warmup_iters = 200
# max_iters = 40000
max_iters = 10000
embed_dim = 512
n_head = 16
n_layer = 4
ff_dim = 1344
theta = 10000
eval_iters = 500
grad_clip = 1.0
save_iters = [500, 5000]

train_data = np.memmap(data_path, dtype = np.uint16, mode = 'r')
valid_data = np.memmap(val_data_path, dtype = np.uint16, mode = 'r')

tokenizer = BPETokenizer.load_from(tokenizer_path)

model = GPT(vocab_size, context_len, embed_dim, n_head, n_layer, ff_dim, theta).to(device)

optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

total_params = sum(p.numel() for p in model.parameters())
print(f"파라미터 수: {total_params}")

val_loss = float("inf")
val_losses = []
val_iters = []

cuda
파라미터 수: 22825472


In [ ]:
pbar = tqdm(range(max_iters))

for i in pbar:
    lr = get_lr(i, learning_rate, warmup_iters, max_iters)

    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    batch_x, batch_y = get_batch(train_data, context_len, batch_size, device)
    optimizer.zero_grad()

    with autocast(device_type = device.type, dtype = torch.bfloat16):
        logits = model(batch_x)
        loss = F.cross_entropy(logits.view(-1, logits.size(-1)), batch_y.view(-1))

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
    optimizer.step()

    if i in save_iters:
        save_path = f"./model_iter_{i}.pt"
        model.save(save_path)
        print('모델이 저장되었습니다.')

    if (i % eval_iters) == 0 or i == max_iters - 1:
        total_loss, total_tokens = evaluate(model, valid_data, context_len, batch_size, device)
        val_loss = total_loss / total_tokens
        val_losses.append(val_loss)
        val_iters.append(i)

    pbar.set_postfix(loss = f'{loss.item():.4f}', val_loss = f'{val_loss:.4f}')

model.save(model_save_path)

  5%|▌         | 500/10000 [01:50<20:42,  7.65it/s, loss=2.5849, val_loss=9.2881]  

모델이 저장되었습니다.


 50%|█████     | 5000/10000 [17:23<12:58,  6.43it/s, loss=1.7238, val_loss=1.6677]   

모델이 저장되었습니다.


100%|██████████| 10000/10000 [34:47<00:00,  4.79it/s, loss=1.4534, val_loss=1.4771]


In [1]:
## 260930

import os, sys
sys.path.append('.')

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

device = get_device()
print(device)

model_path = './model_pretrain.pt'
tokenizer_path = './merge_rules.pkl'

# prompt = 'Once upon a time'
prompt = '<|endoftext|>'

max_new_tokens = 300
temperature = 1.0
num_samples = 3

tokenizer = BPETokenizer.load_from(tokenizer_path)
model = GPT.load_from(model_path, device = device)

for i in range(num_samples):
    print(f"샘플 : {i + 1}")
    story = generate(model, tokenizer, prompt, max_new_tokens, temperature)
    print(story)

cuda
샘플 : 1


Encoding: 100%|██████████| 3/3 [00:00<00:00, 3004.52it/s]



One sunny day, a little girl named Janie went to the park. She saw a big open space where she could run and play without a ball. She ran outside to play, but it was very wet.
As Janie started to play, something unexpected happened. A little dog came running up to her and wagged its tail. The dog jumped up and down, and Janie laughed. The dog seemed to wink at Janie, and head felt warm too.
Then, the big, bouncy ball came close to Jenny's friend, Jane. She threw the ball far away and watched as the dog came back to eat the new ball. Jane was happy and there was no problem, because she had a new friend in targets. Together, they played and laughed all day long, and everyone had fun.

샘플 : 2


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]




Once upon a time, there was an enormousspecice country. On this grand adventure, there lived a brave knight and loud horns.
He was determined to go on an adventure, so he set off towards the escaped from his kingdom.
He recommended rid off and he lifted out his arms light. As he carefully lifted off the ground, he spotted a beautiful treasure chest.
The dragon grabbed the sword with his claws and took a friend. He was so excited and he ran as fast as he could anyway the dragon could hold up the sword.
The dragon spoke and told the knight he'd be my job, to lift him. Sure enough, there was a scale close. She could help him move it!
The knight was delighted and so was his friend. He had comforted the dragon and took him. From that day on, they rolled together and explored the mysterious country.

샘플 : 3


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]





Lily and Tom were playing in the garden. They liked to dig holes and plant flowers. They saw a big onion in the dirt. It had a door on it.
"Wow," Lily said. "Look at what you get for the onion!"
"Yes, really," Tom said. "I want to fill it with beans."
They ran to the bathroom and lifted the door. They filled the bucket with dirt and bubbles. They put the onion inside the door to carry it out.
"Let's put the onion inside," Lily said. "Maybe we can make soup better."
Tom nodded. He wanted to see if there were any peppers in the shed. He was thirsty.
They lifted the onion up and put it in the water. They watched as the onion tipped over.
"Wow," Lily said. "That was fun!"
"Let's do it!" Tom said. "The onion is good!"
They pulled and pushed into the pot. The soup tasted bad. They stirred the soup some more. They had no choice.
"That's not a good soup!" Lily said. "It tastes bad!"
"No, that's bad!" Tom said. "You used it. It tastes bad. They liked to draw on different colors of pastel flo

In [4]:
# 샘플이 얼마나 잘 만들어 졌는지 확인(평가)

import json
import statistics

from openai import OpenAI

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

client = OpenAI()  # OPENAI_API_KEY 환경변수 사용

device = get_device()
print(device)

tokenizer_path = './merge_rules.pkl'
tokenizer = BPETokenizer.load_from(tokenizer_path)

model_paths = {
    500: './model_iter_500.pt',
    5000: './model_iter_5000.pt',
    10000: './model_pretrain.pt',
}

prompt = '<|endoftext|>'
max_new_tokens = 200
temperature = 1.0
num_samples = 10

def evaluate_story(client, story):
    evaluation_prompt = f"""다음 어린이용 스토리를 두 가지 관점에서 1 ~ 5점으로 평가해줘.

    스토리 :
    {story}

    평가관점 :
    1. Coherence(일관성) : 논리적으로 이어지는가, 이야기로서 앞뒤가 맞는가
    2. Grammer(문법) : 문법적으로 올바른 영어인가

    다음 JSON 형식으로 대답해줘.
    {{
        "coherence": <1~5 범위의 정수>,
        "grammer": <1~5 범위의 정수>,
        "comment": "<평가에 대한 간단한 이유>"
    }}"""

    response = client.chat.completions.create(
        model = "gpt-4o-mini",
        messages = [{'role': 'user', 'content': evaluation_prompt}],
        max_tokens = 300,
        response_format = {'type': 'json_object'}
    )

    text = response.choices[0].message.content
    print(text)

    return json.loads(text)

results = {}

for iteration, model_path in model_paths.items():
    print(f"Iteration: {iteration}")

    model = GPT.load_from(model_path, device = device)
    iteration_results = []

    for i in range(num_samples):
        print(f"샘플 {i + 1}")
        story = generate(model, tokenizer, prompt, max_new_tokens, temperature)
        print(f"Story: {story[:200]}")

        scores = evaluate_story(client, story)
        print(f"Scores: {scores}")

        iteration_results.append({
            "story": story,
            "scores": scores
        })

    results[iteration] = iteration_results

print("Summary")

for iteration in model_paths.keys():
    scores_list = [r["scores"] for r in results[iteration]]
    print(f"Iteration {iteration}")

    for key in ["coherence", "grammer"]:
        values = [s[key] for s in scores_list]
        avg = statistics.mean(values)
        std = statistics.stdev(values) if len(values) > 1 else 0
        print(f"{key}: {avg:.2f} +- {std:.2f}")

cuda
Iteration: 500
샘플 1


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]


Story: 

Once upon a time, there lived a girl named Lily. Sue lived in a happy house with her friend, Spot. They had a small house where she was choose. One day, Tim found a big hat on the ground.
"Look, Spo
{
    "coherence": 2,
    "grammer": 2,
    "comment": "The story lacks logical flow and has several disjointed elements that make it hard to follow. Additionally, there are multiple grammatical errors and awkward phrasing throughout the text."
}
Scores: {'coherence': 2, 'grammer': 2, 'comment': 'The story lacks logical flow and has several disjointed elements that make it hard to follow. Additionally, there are multiple grammatical errors and awkward phrasing throughout the text.'}
샘플 2


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]


Story: 
Once upon a time, there was a little girl named Mia. She always support her aunt the factory. One day she went to the church with her mom. They found pretty pictures to hang on the refrigerator. Mia 
{
    "coherence": 2,
    "grammer": 3,
    "comment": "이야기의 전개가 전반적으로 불분명하고, 캐릭터 간의 대화나 사건이 논리적으로 이어지지 않습니다. 문법적으로는 몇 가지 오류가 있지만, 기본적인 의미 전달은 가능하여 3점을 주었습니다."
}
Scores: {'coherence': 2, 'grammer': 3, 'comment': '이야기의 전개가 전반적으로 불분명하고, 캐릭터 간의 대화나 사건이 논리적으로 이어지지 않습니다. 문법적으로는 몇 가지 오류가 있지만, 기본적인 의미 전달은 가능하여 3점을 주었습니다.'}
샘플 3


...

Story: 
Once upon a time, there was a bald man named Tom. Tom had a cane that he liked to chew. One sunny day, he went outside to play with his friend, Sue.
As they were playing, Sue saw a big cat. The cat l
{
    "coherence": 3,
    "grammer": 4,
    "comment": "The story has a general flow, but some transitions and connections could be clearer. For example, the sudden appearance of the sausage plate in the forest feels abrupt. The grammar is mostly correct, but there are some awkward phrases that could be improved."
}
Scores: {'coherence': 3, 'grammer': 4, 'comment': 'The story has a general flow, but some transitions and connections could be clearer. For example, the sudden appearance of the sausage plate in the forest feels abrupt. The grammar is mostly correct, but there are some awkward phrases that could be improved.'}
샘플 9


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]


Story: 
One day, a thin cat and a small dog were playing in the woods. They found a big log near a tree. The cat said, "I want to sit on this log." The dog said, "No, I want to sit on that log." They both wa
{
    "coherence": 3,
    "grammer": 4,
    "comment": "The story has a basic structure and some logical flow, but there are inconsistencies in the events and character reactions that make it somewhat unclear. The grammar is mostly correct, but there are a few awkward phrases and punctuation issues."
}
Scores: {'coherence': 3, 'grammer': 4, 'comment': 'The story has a basic structure and some logical flow, but there are inconsistencies in the events and character reactions that make it somewhat unclear. The grammar is mostly correct, but there are a few awkward phrases and punctuation issues.'}
샘플 10


Encoding: 100%|██████████| 3/3 [00:00<?, ?it/s]


Story: 
Once upon a time, there was a little dog named Spot. Spot liked to play with his friends. One day, Spot saw a big barrel in the yard. He wanted to see what was inside.
Spot went to his friend, the bi
{
    "coherence": 3,
    "grammer": 4,
    "comment": "The story has a basic structure and a simple plot, but the transition from discovering the barrel to it cooking and serving soup is a bit unclear, which affects overall coherence. The grammar is mostly correct, but there are some awkward phrasings like 'an two lunch' that detract from the overall quality."
}
Scores: {'coherence': 3, 'grammer': 4, 'comment': "The story has a basic structure and a simple plot, but the transition from discovering the barrel to it cooking and serving soup is a bit unclear, which affects overall coherence. The grammar is mostly correct, but there are some awkward phrasings like 'an two lunch' that detract from the overall quality."}
Summary
Iteration 500
coherence: 2.00 +- 0.00
grammer: 2.30 +- 0.4

In [5]:
# DPO (Direct Preference Optimization)
# 선호답변과 비선호답변으로 참조모델과 비교해서 선호되는 답변을 생성하는 모델

import sys
sys.path.append('.')

from itertools import cycle

import json
import torch
import torch.nn.functional as F

from torch.utils.data import DataLoader, Dataset

import matplotlib.pyplot as plt

from tqdm import tqdm

from storybot.tokenizer import BPETokenizer
from storybot.model import GPT
from storybot.utils import generate, get_device

In [10]:
device = get_device()
print(device)

data_path = './tiny_stories_dpo.json'
tokenizer_path = './merge_rules.pkl'
pretrain_model_path = './model_pretrain.pt'
dpo_model_save_path = './model_dpo.pt'

context_len = 256
batch_size = 8
learning_rate = 5e-6
beta = 0.1
max_iters = 1000

cuda


In [11]:
class DPODataset(Dataset):
    def __init__(self, data_path, tokenizer, context_len):
        self.tokenizer = tokenizer
        self.context_len = context_len
        self.samples = []

        with open(data_path)as f:
            data = json.load(f)

        for item in data:
            sample = self._create_sample(item['prompt'], item['chosen'], item['rejected'])
            self.samples.append(sample)

    def _pad_and_mask(self, ids, prompt_len):
        mask = [0] * prompt_len + [1] * (len(ids) - prompt_len)

        if len(ids) > self.context_len:
            ids = ids[:self.context_len]
            mask = mask[:self.context_len]

        else:
            pad_len = self.context_len - len(ids)
            ids = ids + [0] * pad_len
            mask = mask + [0] * pad_len

        return ids, mask

    def _create_sample(self, prompt, chosen, rejected):
        prompt_ids = self.tokenizer.encode(prompt)
        chosen_ids = prompt_ids + self.tokenizer.encode(chosen)
        rejected_ids = prompt_ids + self.tokenizer.encode(rejected)

        prompt_len = len(prompt_ids)
        chosen_ids, chosen_mask = self._pad_and_mask(chosen_ids, prompt_len)
        rejected_ids, rejected_mask = self._pad_and_mask(rejected_ids, prompt_len)

        return chosen_ids, chosen_mask, rejected_ids, rejected_mask

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        chosen_ids, chosen_mask, rejected_ids, rejected_mask = self.samples[idx]

        return (
            torch.tensor(chosen_ids, dtype = torch.long),
            torch.tensor(chosen_mask, dtype = torch.long),
            torch.tensor(rejected_ids, dtype = torch.long),
            torch.tensor(rejected_mask, dtype = torch.long),
        )

In [12]:
def get_sequence_logprobs(model, ids, mask):
    logits = model(ids)
    log_probs = F.log_softmax(logits[:, :-1, :], dim = -1)
    labels = ids[:, 1:]

    per_token_logprobs = torch.gather(log_probs, dim = -1, index = labels.unsqueeze(-1)).squeeze(-1)
    masked_logprobs = per_token_logprobs * mask[:, 1:]

    return masked_logprobs.sum(dim = -1)

def compute_dpo_loss(model, ref_model, chosen_ids, chosen_mask, rejected_ids, rejected_mask, beta):
    chosen_logprobs = get_sequence_logprobs(model, chosen_ids, chosen_mask)
    rejected_logprobs = get_sequence_logprobs(model, rejected_ids, rejected_mask)

    with torch.no_grad():
        ref_chosen_logprobs = get_sequence_logprobs(ref_model, chosen_ids, chosen_mask)
        ref_rejected_logprobs = get_sequence_logprobs(ref_model, rejected_ids, rejected_mask)

    logits = beta * ((chosen_logprobs - rejected_logprobs) - (ref_chosen_logprobs - ref_rejected_logprobs))

    return -F.logsigmoid(logits).mean()

In [13]:
tokenizer = BPETokenizer.load_from(tokenizer_path)
dataset = DPODataset(data_path, tokenizer, context_len)
dataloader = DataLoader(dataset, batch_size = batch_size, shuffle = True)

model = GPT.load_from(pretrain_model_path, device = device)
ref_model = GPT.load_from(pretrain_model_path, device = device)
ref_model.eval()
optimizer = torch.optim.AdamW(model.parameters(), lr = learning_rate)

losses = []
data_iter = cycle(dataloader)

Encoding: 100%|██████████| 1/1 [00:00<?, ?it/s]


In [17]:
pbar = tqdm(range(max_iters))

for i in pbar:
    chosen_ids, chosen_mask, rejected_ids, rejected_mask = next(data_iter)
    chosen_ids, chosen_mask = chosen_ids.to(device), chosen_mask.to(device)
    rejected_ids, rejected_mask = rejected_ids.to(device), rejected_mask.to(device)

    loss = compute_dpo_loss(model, ref_model, chosen_ids, chosen_mask, rejected_ids, rejected_mask, beta)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())
    pbar.set_postfix({'loss': f'{loss.item():.4f}'})

model.save(dpo_model_save_path)

100%|██████████| 1000/1000 [02:07<00:00,  7.84it/s, loss=0.0034]


In [19]:
import sys
sys.path.append('.')

import json
import torch
import torch.nn.functional as F

from openai import OpenAI

from storybot.model import GPT
from storybot.tokenizer import BPETokenizer
from storybot.utils import get_device, generate

client = OpenAI()  # OPENAI_API_KEY 환경변수 사용

device = get_device()
print(device)

tokenizer_path = './merge_rules.pkl'
tokenizer = BPETokenizer.load_from(tokenizer_path)

model_paths = {
    'pretrain': './model_pretrain.pt',
    'dpo': './model_dpo.pt'
}

prompt = 'Once upon a time'
num_comparisons = 100
max_new_tokens = 150
temperature = 1.0

def compare_stories(client, story_a, story_b):
    """두 스토리를 비교하여 어느 쪽이 더 해피엔딩에 가까운지 판정"""
    evaluation_prompt = f"""다음의 두 어린이용 스토리를 비교하여, 어느 쪽이 더 해피엔딩인지 판단해줘.
    [Story A]
    {story_a}
    [Story B]
    {story_b}

    어느 쪽의 결말이 더 밝고 행복한지, 또는 내용이 더 희망적인지 판단해줘.
    JSON형식으로 답변: {{
        "winner": "A" or "B" or "tie,
        "reason": "간단한 이유"
    }}
    """

    response = client.chat.completions.create(
        model = 'gpt-4o-mini',
        messages = [{'role': 'user', 'content': evaluation_prompt}],
        max_tokens = 150,
        response_format = {'type': 'json_object'}
    )

    text = response.choices[0].message.content

    return json.loads(text)

model_pretrain = GPT.load_from(model_paths['pretrain'], device = device)
model_dpo = GPT.load_from(model_paths['dpo'], device = device)

results = []
wins = {'pretrain': 0, 'dpo': 0, 'tie': 0}

for i in range(num_comparisons):
    print(f"Comparison {i + 1} / {num_comparisons}")

    story_pretrain = generate(model_pretrain, tokenizer, prompt, max_new_tokens, temperature)
    story_dpo = generate(model_dpo, tokenizer, prompt, max_new_tokens, temperature)

    print(f"[Pretrain]: {story_pretrain[:100]}...")
    print(f"[DPO]: {story_dpo[:100]}...")

    # 편향 방지
    import random

    if random.random() < 0.5:
        story_a, story_b = story_pretrain, story_dpo
        mapping = {'A': 'pretrain', 'B': 'dpo'}

    else:
        story_a, story_b = story_dpo, story_pretrain
        mapping = {'A': 'dpo', 'B': 'pretrain'}

    judgement = compare_stories(client, story_a, story_b)

    winner_label = judgement['winner']

    if winner_label == 'tie':
        winner = 'tie'

    else:
        winner = mapping[winner_label]

    wins[winner] += 1

    print(f'승자: {winner}')
    print(f"이유: {judgement['reason']}")

    results.append({
        'story_pretrain': story_pretrain,
        'story_dpo': story_dpo,
        'winner': winner,
        'reason': judgement['reason']
    })

cuda
Comparison 1 / 100


Encoding: 100%|██████████| 1/1 [00:00<?, ?it/s]


[Pretrain]: Once upon a time, there was a little girl named Thanka. She was unexpected for a pony! She was so pr...
[DPO]: Once upon a time there lived a friendly fox called Bill. His name was Joe. One day, he was walking t...
승자: dpo
이유: Story B portrays a friendly interaction between characters, ending with smiles and laughter, while Story A has conflict and a lack of resolution.
Comparison 2 / 100


Encoding: 100%|██████████| 1/1 [00:00<00:00, 1000.07it/s]


[Pretrain]: Once upon a time, there was a strong boy named Tom. Tom liked to walk and walk to the library. One d...
[DPO]: Once upon a time, there was a stubborn cat. His name was Max. Max liked to play with his ball. But M...
승자: dpo
이유: Story B의 결말은 Max와 Mia가 함께 놀며 즐거운 시간을 보낸 것이기 때문에 더 행복하고 긍정적입니다.
Comparison 3 / 100


...

[Pretrain]: Once upon a time, in a big, green forest, there lived a reindeer. This reindeer was very eager to fi...
[DPO]: Once upon a time, in a small town, there was a mysterious box. Inside the box, there were many yummy...
승자: pretrain
이유: 스토리 A는 친구를 도와주고 함께 행복한 시간을 보내며 긍정적인 관계를 형성하는 내용으로, 더 행복한 결말을 가지고 있습니다.
Comparison 99 / 100


Encoding: 100%|██████████| 1/1 [00:00<?, ?it/s]


[Pretrain]: Once upon a time, there was a smart dog named Max. Max loved to run and play with his friends. One d...
[DPO]: Once upon a time, there was a hairy dog named Bobo. Bobo loved to perform with his friends. They wou...
승자: dpo
이유: Bobo의 공연이 모두에게 즐거움을 주었고, 친구들과 가족들이 함께 clapping하고 행복하게 시간을 보냈기 때문에 더 해피엔딩이라고 볼 수 있습니다.
Comparison 100 / 100


Encoding: 100%|██████████| 1/1 [00:00<00:00, 996.75it/s]


[Pretrain]: Once upon a time, in a small house, there lived a happy cat and a little dog. They had a fun life to...
[DPO]: Once upon a time, there was a little girl named Amy. She had a favorite book about a cat pulled mess...
승자: dpo
이유: Story B has a positive and collaborative ending where Amy and Tim succeed in their task and feel proud of their achievement, promoting themes of friendship and teamwork.


In [20]:
print("일대일 비교 결과")

total = num_comparisons
print(f"사전 학습 모델 승리: {wins['pretrain']:3d} ({wins['pretrain']/total*100:5.1f}%)")
print(f"DPO 모델 승리: {wins['dpo']:3d} ({wins['dpo']/total*100:5.1f}%)")
print(f"무승부: {wins['tie']:3d} ({wins['tie']/total*100:5.1f}%)")

if wins['pretrain'] + wins['dpo'] > 0:
    dpo_winrate = wins['dpo'] / (wins['pretrain'] + wins['dpo']) * 100
    print(f"\n  DPO 모델 승률(무승부 제외): {dpo_winrate:.1f}%")

일대일 비교 결과
사전 학습 모델 승리:  29 ( 29.0%)
DPO 모델 승리:  71 ( 71.0%)
무승부:   0 (  0.0%)

  DPO 모델 승률(무승부 제외): 71.0%
